In [38]:
import cv2
import json
import os
import glob

# --- CONFIG ---
DATASET_BASE = "D:/GitHub/BaseballPitch/modules/pitcher_segmentation/finetuning_dataset"
SPLIT = "val"  # "train" or "val" or "test"
PITCH_TYPE = "FF - Fastball"
# Progress tracking file — records which reviews are completed
REVIEW_PROGRESS_FILE = f"{DATASET_BASE}/review_progress.json"
# --------------

CLASS_NAMES = ['pitcher']
CLASS_COLORS = [(0, 255, 0)]  # Green for pitcher

In [39]:
def _load_review_progress():
    if os.path.exists(REVIEW_PROGRESS_FILE):
        with open(REVIEW_PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {}

def _save_review_progress(progress):
    os.makedirs(os.path.dirname(REVIEW_PROGRESS_FILE), exist_ok=True)
    with open(REVIEW_PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

def _review_progress_key():
    return f"review/{SPLIT}/{PITCH_TYPE}"

def draw_yolo_box(img, label_line):
    """Draw bounding box from YOLO format label"""
    h, w = img.shape[:2]
    label_line = label_line.replace('\\n', '').strip()
    parts = label_line.strip().split()
    if len(parts) < 5:
        return
    
    # YOLO format: class x_center y_center width height
    cls = int(parts[0])
    x_c, y_c, box_w, box_h = map(float, parts[1:5])
    
    # Convert box to pixel coordinates
    x_c *= w
    y_c *= h
    box_w *= w
    box_h *= h
    
    x1 = int(x_c - box_w/2)
    y1 = int(y_c - box_h/2)
    x2 = int(x_c + box_w/2)
    y2 = int(y_c + box_h/2)
    
    # Get class info
    class_name = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"Class {cls}"
    color = CLASS_COLORS[cls] if cls < len(CLASS_COLORS) else (255, 255, 255)
    
    # Draw bounding box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    
    # Draw label background
    label_text = class_name
    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cv2.rectangle(img, (x1, y1 - text_h - 10), (x1 + text_w + 10, y1), color, -1)
    cv2.putText(img, label_text, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

def review_labels():
    progress = _load_review_progress()
    progress_key = _review_progress_key()

    # Load set of already-reviewed filenames
    entry = progress.get(progress_key, {})
    reviewed_files = set(entry.get("reviewed", []))

    img_dir = os.path.join(DATASET_BASE, "images", SPLIT, PITCH_TYPE)
    label_dir = os.path.join(DATASET_BASE, "labels", SPLIT, PITCH_TYPE)
    
    # Get all images, then filter to only unreviewed
    all_images = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    images = [p for p in all_images if os.path.basename(p) not in reviewed_files]
    
    if not images:
        print(f"⏭️  All {len(all_images)} images already reviewed in {SPLIT}/{PITCH_TYPE}")
        print(f"   (Reviewed: {len(reviewed_files)}, Deleted so far: {entry.get('deleted', 0)})")
        return
    
    print(f"Found {len(images)} new images to review in {SPLIT}/{PITCH_TYPE}")
    if reviewed_files:
        print(f"  (Already reviewed: {len(reviewed_files)}, previously deleted: {entry.get('deleted', 0)})")
    print("\nControls:")
    print("  SPACE/RIGHT ARROW/S - Next image")
    print("  LEFT ARROW/A - Previous image")
    print("  D - Delete current image and label")
    print("  Q/ESC - Quit")
    print("\n" + "="*50)
    
    idx = 0
    deleted_count = 0
    newly_reviewed = []
    
    while idx < len(images):
        img_path = images[idx]
        img_name = os.path.basename(img_path)
        label_path = os.path.join(label_dir, img_name.replace('.jpg', '.txt'))
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Failed to load {img_path}")
            idx += 1
            continue
        
        # Draw bounding boxes if label exists
        has_label = False
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    draw_yolo_box(img, line)
                has_label = True
        
        # Add info overlay
        info_text = f"[{idx+1}/{len(images)}] {img_name}"
        if not has_label:
            info_text += " (NO LABEL)"
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 1)
        
        # Show image
        cv2.imshow("Label Review", img)
        
        # Wait for key (use waitKeyEx for reliable arrow keys on Windows)
        key = cv2.waitKeyEx(0)

        # Quit (partial — save what we've reviewed so far)
        if key in (ord('q'), ord('Q'), 27):  # Q or ESC
            break
        # Delete current image and label
        elif key in (ord('d'), ord('D')):  # Delete
            os.remove(img_path)
            if os.path.exists(label_path):
                os.remove(label_path)
            # Track as reviewed (deleted files won't reappear anyway)
            newly_reviewed.append(img_name)
            images.pop(idx)
            deleted_count += 1
            print(f"Deleted: {img_name}")
            idx = max(0, idx)  # Stay at same position
        # Next image
        elif key in (
            32,               # Space
            2555904,          # Right arrow (Windows)
            83,               # Right arrow (Linux/X11)
            ord('s'), ord('S')  # 'S' key
        ):
            # Mark current image as reviewed when moving past it
            newly_reviewed.append(img_name)
            idx = min(idx + 1, len(images) - 1)
        # Previous image
        elif key in (
            2424832,          # Left arrow (Windows)
            81,               # Left arrow (Linux/X11)
            8,                # Backspace
            ord('a'), ord('A')  # 'A' key
        ):
            idx = max(idx - 1, 0)
    
    cv2.destroyAllWindows()

    # Merge newly reviewed into progress
    reviewed_files.update(newly_reviewed)
    prior_deleted = entry.get("deleted", 0)

    progress[progress_key] = {
        "reviewed": sorted(reviewed_files),
        "deleted": prior_deleted + deleted_count,
    }
    _save_review_progress(progress)

    print(f"\nReview session complete!")
    print(f"  Newly reviewed: {len(newly_reviewed)}")
    print(f"  Deleted this session: {deleted_count}")
    print(f"  Total reviewed: {len(reviewed_files)}")
    print(f"  Total deleted:  {prior_deleted + deleted_count}")

In [37]:
review_labels()

Found 12402 new images to review in train/FF - Fastball
  (Already reviewed: 37026, previously deleted: 271)

Controls:
  SPACE/RIGHT ARROW/S - Next image
  LEFT ARROW/A - Previous image
  D - Delete current image and label
  Q/ESC - Quit

Deleted: PitchType-FF_Zone-11_PlayID-9d1d69af-d560-3473-9a74-2fba8b2e1911_Date-2025-05-30_0000.jpg
Deleted: PitchType-FF_Zone-11_PlayID-9d1d69af-d560-3473-9a74-2fba8b2e1911_Date-2025-05-30_0004.jpg
Deleted: PitchType-FF_Zone-11_PlayID-9d1d69af-d560-3473-9a74-2fba8b2e1911_Date-2025-05-30_0008.jpg
Deleted: PitchType-FF_Zone-11_PlayID-9d1d69af-d560-3473-9a74-2fba8b2e1911_Date-2025-05-30_0012.jpg
Deleted: PitchType-FF_Zone-11_PlayID-9d1d69af-d560-3473-9a74-2fba8b2e1911_Date-2025-05-30_0016.jpg
Deleted: PitchType-FF_Zone-11_PlayID-aa8602c9-efa2-4598-b6a4-b1f0813ed02f_Date-2023-09-29_0000.jpg
Deleted: PitchType-FF_Zone-11_PlayID-aa8602c9-efa2-4598-b6a4-b1f0813ed02f_Date-2023-09-29_0004.jpg
Deleted: PitchType-FF_Zone-11_PlayID-aa8602c9-efa2-4598-b6a4-b1f081

In [34]:
# --- Review Progress Summary ---
progress = _load_review_progress()

review_entries = {k: v for k, v in progress.items() if k.startswith("review/")}

if not review_entries:
    print("No review progress recorded yet.")
else:
    total_reviewed = sum(len(v.get("reviewed", [])) for v in review_entries.values())
    total_deleted = sum(v.get("deleted", 0) for v in review_entries.values())

    print(f"{'='*60}")
    print(f"Review Progress Summary")
    print(f"{'='*60}")
    print(f"  Total reviewed: {total_reviewed}")
    print(f"  Total deleted:  {total_deleted}")
    print(f"{'='*60}")

    for key in sorted(review_entries):
        entry = review_entries[key]
        reviewed = len(entry.get("reviewed", []))
        deleted = entry.get("deleted", 0)
        parts = key.split("/", 2)  # review / split / pitch_type
        split_name = parts[1] if len(parts) > 1 else "?"
        pitch_name = parts[2] if len(parts) > 2 else "?"
        print(f"  [{split_name}] {pitch_name}  —  "
              f"reviewed: {reviewed}, deleted: {deleted}")

Review Progress Summary
  Total reviewed: 105325
  Total deleted:  1300
  [train] CH - Changeup  —  reviewed: 61851, deleted: 828
  [train] FF - Fastball  —  reviewed: 37026, deleted: 271
  [val] CH - Changeup  —  reviewed: 6448, deleted: 201
